In [1]:
import tomllib
from pathlib import Path

import pandas as pd

from metadata_converter.flat_data.transform.clean import strip_header_whitespace
from metadata_converter.flat_data.extract import extract_data
from metadata_converter.flat_data.config import FlatDataConfig


In [2]:
file_path = Path("/home/japlan001/PycharmProjects/paper/data/input/continuous_reporting")
config_path = Path("/home/japlan001/PycharmProjects/paper/configs/continuous_reporting.toml")
output_path = Path("./filtered_continuous_reporting")

# ensure that the outputpath exists
output_path.mkdir(parents=True, exist_ok=True)

In [3]:
config = tomllib.loads(config_path.read_text(encoding="utf-8"))

# drop the plugin entries as they will prevent building the config via pydantic as there are files and folders requested, which are not needed
config["cleaning"].pop("plugin_dir")
config["cleaning"].pop("plugin_name")

'combine_month_year_rows.py'

In [4]:
def append_single(append_list: list, item: any) -> None:
    """Append all single items from item to create a flat list."""
    if isinstance(item, list):
        for subitem in item:
            append_single(append_list, subitem)
    elif isinstance(item, dict):
        for value in item.values():
            append_single(append_list, value)
    else:
        append_list.append(item)

In [5]:
def extract_requested_headers(config: dict) -> list:
    # we need these two for the plugin to work
    requested_headers = []
    keys = ["combined_columns", "split_fields", "mapping"]

    for key in keys:
        for item in pd.json_normalize(config.get(key), sep=".").values.flatten():
            append_single(requested_headers, item)

    # filter out obviously invalid header
    requested_headers = [header for header in requested_headers if not str(header).startswith("Literal:") and str(header) not in ["@id", "Person", "ScholarlyArticle", "Thing", "PropertyValue"]]
    return requested_headers

In [6]:
requested_headers = extract_requested_headers(config)
requested_headers

['First Name',
 'Surname',
 'Authors',
 'name',
 'First Name',
 'Surname',
 'Personal identifier e.g. ORCID (optional)',
 'Type of publication',
 'PID of deposited publication',
 'Link to publication (if no PID of publication)',
 'Type of PID (repository)',
 'Link to publication (if no PID of publication)',
 'Title of the publication (for book chapters: title of chapter not of book)',
 'Type of publication',
 'Authors',
 'Periodical',
 'Title of the journal or equivalent',
 'ISSN/ eSSN (if book insert ISBN)',
 'Organization',
 'Publisher',
 'Month/ Year of publication',
 'License type',
 'Dataset',
 'Description of datasets',
 'PID of deposited dataset',
 'URL to repository (if required by WP is the repository federated under EOSC?)',
 'Type of PID',
 'URL to repository (if required by WP is the repository federated under EOSC?)',
 'What license is the data licensed under?',
 'Type of result',
 'Description',
 'PID (if available)',
 'URL to repository landing page for the result servic

In [7]:
def extract_requested_sheets(config: dict) -> list:
    return list(config["mapping"].keys())

In [8]:
requested_sheets = extract_requested_sheets(config)
requested_sheets

['Researchers', 'Publications', 'Dataset', 'Other_Results']

In [9]:
for file in file_path.glob("*.xlsx"):
    conf = FlatDataConfig.model_validate(config)
    sheets = extract_data(conf, file)

    # ignore all sheets, that are not used
    drop_sheets = set(sheets.keys()) - set(requested_sheets)
    for sheet in drop_sheets:
        sheets.pop(sheet)

    for sheet, df in sheets.items():
        df = strip_header_whitespace(df)
        print(df)
        sheets[sheet] = df.loc[:, df.columns.isin(requested_headers)]

    with pd.ExcelWriter(output_path / file.name) as writer:
        for sheet, df in sheets.items():
            df.to_excel(writer, sheet_name=sheet, index=False)

   [Beneficiary No]        First Name            Surname         Gender  \
0          28-SOTON          Geoffrey            Neumann              M   
1             18-SU             Julie            Lemoine              F   
2          21-UNIPD  Giorgio Vittorio              Visco              M   
3            8-AZTI           Guillem              Chust              M   
4            2-CNRS           Pauline            Narvaez              F   
5            2-CNRS              Blai  Vidiella Rocamora              M   
6            16-DTU            Ken H.           Andersen              M   
7          17-CIRAD          Matthieu            Barbier              M   
8            29-NOC           Shubham            Krishna              M   
9            2-CNRS         Alexandre              Génin              M   
10         21-UNIPD              Kobe            Simoens  Please select   
11       26-UNANTES            Lilian           Marchand  Please select   
12           20-CEA      

In [10]:
conf = FlatDataConfig.model_validate(config)
#data = extract_data(config)

In [11]:
conf

FlatDataConfig(source_type='flat_data', extractor=ExcelExtractorConfig(input=PosixPath('data/input/continuous_reporting'), sheet_name=['Researchers', 'Publications', 'Dataset', 'Other_Results'], header={'Researchers': 2, 'Publications': 4, 'Dataset': 3, 'Other_Results': 3}, skiprows={'Researchers': [3, 4], 'Publications': [5, 6], 'Dataset': [4, 5], 'Other_Results': [4, 5]}), cleaning=CleaningConfig(strip_header_whitespace=True, strip_cell_whitespace=True, sentinels_to_na=True, empty_sentinels=['', 'N/A', 'n/a', '-', 'Please select'], placeholders_to_na=True, placeholder_pattern='^.*\\[.*\\]$', plugin_dir=None, plugin_name=None, plugins=[]), output_dir=PosixPath('data/loaded_base/continuous_reporting'), provenance_dir=PosixPath('data/provenance/load/continuous_reporting'), mapping={'Researchers': {'type': 'Person', 'id': '@id', 'name': 'name', 'givenName': 'First Name', 'familyName': 'Surname', 'identifier': 'Personal identifier e.g. ORCID (optional)'}, 'Publications': {'type': 'Scholar